<img src="https://www.hazfundacion.org/wp-content/uploads/2022/11/U-Carlos-III.png" width="300" alt="Universidad Carlos III de Madrid">

**Máster Universitario en Robótica y Automatización**

Asignatura: Procesamiento de Imágenes por Computador

Curso: 2026-27

Autores:
- Anthony Manzueta Difo
- Stefan Oshchypok
- Marina Ruiz Izu

Proyecto: **Reconocimiento de ocupación de plazas de aparcamiento mediante fine-tuning de redes preentrenadas**

# Problema a tratar

## Motivación

Conocer la disponibilidad de plazas libres es útil para la gestión de los aparcamientos y para reducir el tiempo y el tráfico que genera la búsqueda de plaza. Con la ocupación en tiempo real, los conductores ahorran tiempo de búsqueda.

Desde el punto de vista empresarial, la visión por computador aprovecha las cámaras de seguridad ya instaladas. Evita el coste de un sensor por plaza (en torno a 300-600 $ por plaza en los sistemas tradicionales) y la obra asociada. Además de la ocupación en tiempo real, permite obtener datos de uso (ocupación por franja horaria, tiempos de estancia) que ayudan a dimensionar y optimizar el aparcamiento. En empresas con parking en sus oficinas, mejora la experiencia de los empleados y reduce el tiempo y las emisiones de la búsqueda de plaza.

## Objetivo

Detectar plazas de aparcamiento libres y ocupadas mediante una cámara exterior. Dada una imagen de aparcamiento, el sistema localiza cada plaza y la clasifica como libre u ocupada, devolviendo para cada una un valor de confianza.

# Etapa 1: Preparación del Dataset

Aqui vamos a procesar un dataset [MetaPKLot-Dataset](https://huggingface.co/datasets/RajChauhan2911/MetaPKLot-Dataset) para crear nuestro mas pequeño y adaptado al problema enforntado.

El nuevo dataset creado debe ser subido al nuevo repositorio de [HuggingFace](https://huggingface.co/datasets/StafLoker/Meta-PKLot-mini).

## Imports

In [11]:
# !pip install -q huggingface_hub

import glob
import json
import os
import tarfile
from collections import Counter

from huggingface_hub import snapshot_download

## Config

In [6]:
CONFIG = {
    'dataset': {
        'source': {
            'repo': 'RajChauhan2911/MetaPKLot-Dataset',
            'path': '../data/MetaPKLot', # /content/data/MetaPKLot for colab
        },
        'mini': {
            'repo': 'StafLoker/Meta-PKLot-mini',
            'path': '../data/Meta-PKLot-mini', # /content/data/Meta-PKLot-mini for colab
        },
    }
}

## Dataset

### Download

In [7]:
snapshot_download(
    repo_id=CONFIG['dataset']['source']['repo'],
    repo_type='dataset',
    local_dir=CONFIG['dataset']['source']['path'],
    token=os.environ.get('HF_TOKEN'),
)

source_path = CONFIG['dataset']['source']['path']
num_files = 0
total_bytes = 0

for root, _, files in os.walk(source_path):
    num_files += len(files)
    total_bytes += sum(os.path.getsize(os.path.join(root, f)) for f in files)

print(f"Dataset size: {total_bytes / 1024**3:.2f} GB ({num_files} files)")

Fetching 16604 files: 100%|██████████| 16604/16604 [36:54<00:00,  7.50it/s]


Dataset size: 4.63 GB (49815 files)


### Estructura de las anotaciones del dataset original

#### COCO

Las anotaciones están en formato **COCO** (JSON). Un fichero COCO tiene cinco secciones principales:

```json
{
  "info": {}, // Información del dataset: versión, descripción, fecha, etc.
  "licenses": [], // Licencias de las imágenes.
  "images": [], // Información de cada imagen.
  "categories": [], // Clases de objetos que reconoce el modelo.
  "annotations": [] // Anotaciones de los objetos presentes en las imágenes.
}
```

#### MetaPKLot

MetaPKLot reúne tres datasets: **PKLot**, **CNRPark-EXT** y **PLds**. Todos siguen la estructura COCO y se reparten en `annotations/`:

```
annotations/{original,cropped}/{cars,spots}/<dataset>/<subset>_<tipo>.tar.xz
```

Cada `.tar.xz` contiene un único JSON con las secciones `categories`, `climates`, `images`, `annotations` e `info`.

- **`spots`:** plazas, con las categorías `0 = empty` y `1 = occupied`.
- **`cars`:** vehículos, con la categoría `1 = car`.
- **`climates`:** idéntico en los tres datasets: `0 Overcast`, `1 Cloudy`, `2 Sunny`, `3 Rainy`, `4 Snowy`, `5 Clearnight`.

Campos comunes de cada imagen (`images`):

```json
{
  "images": [
    {
      "id": 0,
      "file_name": "...", // Ruta relativa a la raíz del dataset
      "width": 0,
      "height": 0,
      "annotationsRectangle": [x, y, w, h], // Región con anotación completa: fuera de ella no hay negativos fiables
      "date": [año, mes, día],
      "time": [hora, minuto, segundo],
      "climate": 0, // id de `climates`
      "dataset": "...",
      "subset": "..."
    },
    ...
  ]
}
```

Campos comunes de cada anotación de plaza (`annotations` de `spots`):

```json
{
  "annotations": [
    {
      "id": 0,
      "image_id": 0, // Imagen a la que pertenece
      "category_id": 1, // 0 = empty, 1 = occupied
      "segmentation": [[x1, y1, x2, y2, ...]], // Polígono de la plaza
      "bbox": [x, y, w, h], // Esquina superior izquierda, ancho y alto
      "area": 0.0,
      "iscrowd": 0,
      "car_id": 0, // Vehículo que la ocupa (-1: ninguno, -2: algo que no es un coche)
      "best_rotated_rect": [[cx, cy], [w, h], ángulo] // Rectángulo rotado de la plaza
    },
    ...
  ]
}
```

A continuación, un registro real de plazas de cada dataset (`original/spots`, polígonos recortados).

#### PKLot

3 subsets (`pucpr`, `ufpr04`, `ufpr05`), 12 400 imágenes y 1 199 857 plazas. Sunny, Cloudy y Rainy.

```json
{
  "images": [
    {
      "id": 11000002330,
      "file_name": "PKLot/PUCPR/Sunny/2012-09-11/2012-09-11_15_16_58.jpg",
      "width": 1280, "height": 720,
      "annotationsRectangle": [0, 140, 1280, 580],
      "date": [2012, 9, 11], "time": [15, 16, 58],
      "climate": 2, "dataset": "pklot", "subset": "pucpr"
    },
    ...
  ],
  "annotations": [
    {
      "id": 11000000000,
      "image_id": 11000002330, // id de la imagen de `images`
      "category_id": 1,
      "segmentation": [[278, 230, 290, 186, 324, 185, 308, 230]],
      "bbox": [278, 185, 46, 45], "area": 1417.0, "iscrowd": 0,
      "car_id": 11000012277,
      "best_rotated_rect": [[299.03, 205.51], [32.27, 58.68], 14.93]
    },
    ...
  ]
}
```

- Imágenes de 1280×720 y coordenadas enteras.
- `annotationsRectangle` depende del subset: `[0, 140, 1280, 580]` en PUCPR y `[0, 0, 1280, 720]` en UFPR04.
- `car_id` negativo: `-1` en 656 269 plazas y `-2` en 106.

#### CNRPark-EXT

9 subsets (`camera1` … `camera9`), 4 073 imágenes y 165 530 plazas. Sunny, Overcast y Rainy.

```json
{
  "images": [
    {
      "id": 21000000403,
      "file_name": "CNRPark-EXT/camera1/SUNNY/2015-11-12/2015-11-12_07_09_00.jpg",
      "width": 1000, "height": 750,
      "annotationsRectangle": [78, 158, 922, 592],
      "date": [2015, 11, 12], "time": [7, 9, 0],
      "climate": 2, "dataset": "cnr", "subset": "camera1"
    },
    ...
  ],
  "annotations": [
    {
      "id": 21000000000,
      "image_id": 21000000403, // id de la imagen de `images`
      "category_id": 0,
      "segmentation": [[391.4, 328.6, 435.0, 357.4, 347.0, 419.5, 296.0, 392.4]],
      "bbox": [296.0, 328.6, 139.0, 90.9], "area": 5544.0, "iscrowd": 0,
      "car_id": -1,
      "best_rotated_rect": [[366.25, 376.02], [50.88, 135.0], 56.23]
    },
    ...
  ]
}
```

- Imágenes de 1000×750 y coordenadas decimales.
- La carpeta del clima va en mayúsculas (`SUNNY`).
- `car_id` negativo: `-1` en 78 009 plazas y `-2` en 32.

#### PLds

4 subsets (`isshk`, `qridr`, `vmlix`, `vxusd`), 8 188 imágenes y 104 728 plazas. Sunny, Overcast, Rainy, Snowy y Clearnight. **Las imágenes no están en disco, solo las anotaciones.**

```json
{
  "images": [
    {
      "id": 31000000000,
      "file_name": "PLds/isshk/Daytime/Snow/2016-02-18/isshk_0475.jpg",
      "width": 1920, "height": 1080,
      "annotationsRectangle": [432, 200, 1342, 910],
      "date": [2016, 2, 18], "time": [7, 30, 27],
      "climate": 4, "dataset": "plds", "subset": "isshk"
    },
    ...
  ],
  "annotations": [
    {
      "id": 31000000000,
      "image_id": 31000000000, // id de la imagen de `images`
      "category_id": 0,
      "segmentation": [[918.3, 915.1, 904.0, 1080.0, 1409.0, 1080.0, 1392.3, 919.1]],
      "bbox": [904.0, 915.1, 505.0, 164.9], "area": 79694.0, "iscrowd": 0,
      "car_id": -1,
      "best_rotated_rect": [[1156.5, 997.55], [505.0, 164.9], -0.0]
    },
    ...
  ]
}
```

- La resolución cambia según el subset: 1920×1080 en `isshk` y 1280×720 en `vmlix`.
- La ruta incluye `Daytime/Snow`, pero el campo `climate` dice `Snowy`.
- Coordenadas decimales. `car_id` negativo: solo `-1` (60 421 plazas).

#### Para unificar

- Los `id` llevan un prefijo por dataset (`11…`, `21…`, `31…`), así que se pueden juntar sin colisiones.
- El clima hay que leerlo de `climate`, no de la ruta: cambian la profundidad y las mayúsculas.
- `spots` y `cars` no comparten las mismas imágenes: sin coches anotados, la imagen no aparece en `cars`.
- La tabla «Climate Metadata» del `README.md` de MetaPKLot (ids 1-6, con `no_rain` y `undefined`) no coincide con los JSON (ids 0-5, con `Cloudy` y `Clearnight`). Hay que fiarse de `climates`.

#### `original` frente a `cropped`

Mismas imágenes y mismas anotaciones (`id`, `file_name`, `category_id`, `car_id`). Solo cambia la geometría: `cropped` está recortado a `annotationsRectangle`.

|                                             | `original`                        | `cropped`                                                     |
| ------------------------------------------- | --------------------------------- | ------------------------------------------------------------- |
| `width` / `height`                          | imagen completa                   | recorte (PUCPR: 1280×580 en vez de 1280×720)                  |
| `annotationsRectangle`                      | sí                                | no (salvo en `vmlix` y `vxusd`, donde empieza en `[0, 0, …]`) |
| `bbox`, `segmentation`, `best_rotated_rect` | coordenadas de la imagen completa | desplazadas al origen del recorte (PUCPR: `y` baja 140)       |
| `dataset` / `subset`                        | `pklot` / `pucpr`                 | `PKLot` / `pucpr_spots` (`vmlix` y `vxusd` no cambian)        |
| `file_name`                                 | imagen completa                   | **la misma ruta**                                             |

Las imágenes que hay en disco son las completas, y `file_name` de `cropped` apunta a ellas, así que sus coordenadas no encajan. Usaremos `original`.

En los subsets donde el recorte ya cubre toda la imagen (UFPR04, UFPR05, `camera2` … `camera8`) el tamaño no cambia, pero la `bbox` difiere en ±1 píxel (en PUCPR, `[278, 185, 46, 45]` frente a `[278, 45, 47, 46]`). No he comprobado cómo se calcula.

#### Carpetas auxiliares

- **`removed_images/`:** `pklot.txt` (17), `cnrpark-ext.txt` (205) y `plds.txt` (157). Rutas de imágenes eliminadas: corruptas en PKLot, mala iluminación en CNRPark-EXT, timestamps repetidos o zoom inesperado en PLds. El nº de imágenes de `spots` (12 400 y 4 073) coincide con los `.jpg` en disco.
- **`non_working_days/`:** un `days_ignore` por subset, con líneas como `2012-09-15 / Saturday`. Fines de semana y festivos.
- **`non_representative_days/`:** un `non_representative_days` por subset, con líneas como `2016-02-18 / day with low occupance`. Días sin trabajo, de poca duración o de poca ocupación. El README recomienda consultarlos al reproducir experimentos.
- **`readme_images/` y `tools/`:** ilustraciones del README y herramientas de marcado, validación y conversión de PLds. No se usan en el proyecto.

### Etiquetas disponibles

Para cada fichero de anotaciones se muestran sus secciones, sus categorías y cuántas anotaciones hay de cada una.

In [12]:
annotations_glob = os.path.join(CONFIG['dataset']['source']['path'], 'annotations', '**', '*.tar.xz')
annotation_files = sorted(glob.glob(annotations_glob, recursive=True))

if not annotation_files:
    raise FileNotFoundError(f"No annotation files found: {annotations_glob}")

for path in annotation_files:
    with tarfile.open(path, 'r:xz') as tar:
        member = next(m for m in tar if m.name.endswith('.json'))
        coco = json.load(tar.extractfile(member))

    category_names = {c['id']: c['name'] for c in coco['categories']}
    counts = Counter(category_names[a['category_id']] for a in coco['annotations'])

    print(os.path.relpath(path, CONFIG['dataset']['source']['path']))
    print(f"  sections:    {list(coco.keys())}")
    print(f"  images:      {len(coco['images'])}")
    print(f"  categories:  {category_names}")
    print(f"  annotations: {dict(counts)}")

annotations/cropped/cars/CNRPark-EXT/cnr-camera-1_cars.tar.xz
  sections:    ['categories', 'climates', 'images', 'annotations', 'info']
  images:      452
  categories:  {1: 'car'}
  annotations: {'car': 10630}
annotations/cropped/cars/CNRPark-EXT/cnr-camera-2_cars.tar.xz
  sections:    ['categories', 'climates', 'images', 'annotations', 'info']
  images:      457
  categories:  {1: 'car'}
  annotations: {'car': 4791}
annotations/cropped/cars/CNRPark-EXT/cnr-camera-3_cars.tar.xz
  sections:    ['categories', 'climates', 'images', 'annotations', 'info']
  images:      450
  categories:  {1: 'car'}
  annotations: {'car': 8129}
annotations/cropped/cars/CNRPark-EXT/cnr-camera-4_cars.tar.xz
  sections:    ['categories', 'climates', 'images', 'annotations', 'info']
  images:      449
  categories:  {1: 'car'}
  annotations: {'car': 10989}
annotations/cropped/cars/CNRPark-EXT/cnr-camera-5_cars.tar.xz
  sections:    ['categories', 'climates', 'images', 'annotations', 'info']
  images:      45